In [29]:
import os
import re
import json
import pandas as pd
import numpy as np
from dfft.utils.files import get_path


In [30]:

# Helper functions
def extract_time(d):
    m = re.match(r"(\d{8}_\d{6})", os.path.basename(d))
    return m.group(1) if m else ""

def extract_learning_rate(exp_dir_name):
    m = re.search(r"_lr_([0-9]*\.?[0-9]+(?:e-?[0-9]+)?)", exp_dir_name, re.IGNORECASE)
    if not m:
        return None
    return float(m.group(1))

def get_latest_experiments_by_lr(base_dir):
    if not os.path.isdir(base_dir):
        return []
    
    latest_by_lr = {}
    for d in os.listdir(base_dir):
        exp_dir = os.path.join(base_dir, d)
        if not os.path.isdir(exp_dir):
            continue
            
        lr = extract_learning_rate(d)
        if lr is None:
            continue
            
        ts = extract_time(d)
        if lr not in latest_by_lr or ts > latest_by_lr[lr][0]:
            latest_by_lr[lr] = (ts, exp_dir)
            
    return [(lr, latest_by_lr[lr][1]) for lr in sorted(latest_by_lr.keys())]

def get_latest_exp(base_dir):
    if not os.path.isdir(base_dir):
        return None
        
    exp_dirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    if not exp_dirs:
        return None
        
    exp_dirs = sorted(exp_dirs, key=extract_time, reverse=True)
    return exp_dirs[0]

In [31]:
# 1. NonTSFM All Results
families = ["linear", "mlp", "lstm", "gru", "tcn"]
variants = ["small", "base", "large"]
loss_variants = ["", "-decision"]

nontsfm_models = []
for family in families:
    for variant in variants:
        for losser in loss_variants:
            nontsfm_models.append(f"{family}-{variant}{losser}")

all_nontsfm_results = []
for model in nontsfm_models:
    base_dir = get_path(f"nontsfm\\{model}\\NEMA")
    for learning_rate, exp_dir in get_latest_experiments_by_lr(base_dir):
        try:
            with open(exp_dir + "/results.json", 'r') as f:
                data = json.load(f)
            df_parquet = pd.read_parquet(exp_dir + "/results.parquet")
            
            sums = df_parquet.groupby('split')[['true_profit', 'real_profit']].sum()
            regret = (sums['true_profit'] - sums['real_profit']) / sums['true_profit']
            regret = (regret * 100).round(1)
            
            all_nontsfm_results.append({
                'model': model,
                'learning_rate': learning_rate,
                'train_mse': data.get('final_train_results', {}).get('mse', np.nan),
                'val_mse': data.get('final_val_results', {}).get('mse', np.nan),
                'test_mse': data.get('test_results', {}).get('mse', np.nan),
                'train_profit': sums['real_profit'].get('train', np.nan),
                'val_profit': sums['real_profit'].get('val', np.nan),
                'test_profit': sums['real_profit'].get('test', np.nan),
                'train_regret': regret.get('train', np.nan),
                'val_regret': regret.get('val', np.nan),
                'test_regret': regret.get('test', np.nan)
            })
        except Exception as e:
            pass

df_nontsfm = pd.DataFrame(all_nontsfm_results)
if not df_nontsfm.empty:
    df_nontsfm = df_nontsfm.sort_values(['model', 'learning_rate']).reset_index(drop=True)
df_nontsfm

,model,learning_rate,train_mse,val_mse,test_mse,train_profit,val_profit,test_profit,train_regret,val_regret,test_regret
0,gru-base,0.0001,5241.560059,2993.410400,7271.918945,9501.273124,2473.996786,7096.971164,95.4,95.8,92.1
1,gru-base,0.0010,3142.851807,1738.502686,4575.964355,48889.849398,14205.324733,16637.866053,76.3,75.7,81.4
2,gru-base,0.0100,1515.734253,1182.068237,1939.214600,95374.966831,25477.041129,38715.347393,53.7,56.4,56.7
3,gru-base-decision,0.0001,5602.909180,3247.327393,7703.962402,96989.067372,24774.735697,41103.571406,52.9,57.6,54.0
4,gru-base-decision,0.0010,5573.914062,3227.353027,7670.522461,55824.016554,16402.023797,23569.352887,72.9,71.9,73.6
...,...,...,...,...,...,...,...,...,...,...,...
85,tcn-small,0.0010,1057.732544,1166.446289,1289.263916,95064.206577,25703.047966,42702.511533,53.9,56.0,52.2
86,tcn-small,0.0100,1025.333374,1160.054810,1283.133423,99497.809770,26268.149654,42530.438062,51.7,55.0,52.4
87,tcn-small-decision,0.0001,5862.539551,3390.809082,8048.729004,45369.014106,13759.961229,18755.377420,78.0,76.4,79.0
88,tcn-small-decision,0.0010,5342.592773,3146.291016,7316.778320,68767.155492,18782.831636,28516.318275,66.6,67.9,68.1


In [32]:
# 2. NonTSFM Filtered (Best val_profit)
if not df_nontsfm.empty:
    df_nontsfm_best = df_nontsfm.loc[df_nontsfm.groupby('model')['val_profit'].idxmax()].copy()
    df_nontsfm_best = df_nontsfm_best.drop(columns=['learning_rate']).reset_index(drop=True)
else:
    df_nontsfm_best = pd.DataFrame()

df_nontsfm_best

,model,train_mse,val_mse,test_mse,train_profit,val_profit,test_profit,train_regret,val_regret,test_regret
0,gru-base,1515.734253,1182.068237,1939.214600,95374.966831,25477.041129,38715.347393,53.7,56.4,56.7
1,gru-base-decision,5555.891113,3214.285400,7648.048340,97106.125293,26445.512585,42528.262855,52.9,54.7,52.4
2,gru-large,1215.056274,1174.053711,1561.175049,97550.600561,25300.313691,40576.970076,52.7,56.7,54.6
3,gru-large-decision,5580.940430,3232.219971,7679.362793,93636.371264,26545.810577,42162.720163,54.6,54.6,52.8
4,gru-small,1561.716431,1198.576050,2026.640015,94698.833076,23956.951658,37924.494033,54.0,59.0,57.6
5,gru-small-decision,5623.469727,3263.029541,7732.319824,70937.478728,18309.978915,25383.666889,65.6,68.7,71.6
6,linear-base,775.770630,1275.815918,1428.953369,128118.073459,18514.400998,34697.245078,37.8,68.3,61.2
7,linear-base-decision,57725.015625,25483.945312,83132.804688,100338.874446,24654.226925,39852.299595,51.3,57.8,55.4
8,linear-large,810.783813,1279.381348,1409.586914,117574.016966,16652.952619,29996.760974,42.9,71.5,66.4
9,linear-large-decision,53376.886719,23513.183594,76222.101562,97624.396506,23919.463119,41323.869482,52.6,59.1,53.8


In [33]:
# 3. TSFM Models Results
experiments_path = get_path("results\\experiments")
if os.path.isdir(experiments_path):
    tsfm_models = [d for d in os.listdir(experiments_path) if os.path.isdir(os.path.join(experiments_path, d))]
else:
    tsfm_models = []

all_tsfm_results = []
for model in tsfm_models:
    if "time" in model.lower():
        continue
    base_dir = os.path.join(experiments_path, model, "NEMA")
    exp_dir = get_latest_exp(base_dir)
    # print(f"Processing model: {model}, experiment dir: {exp_dir}")
    if not exp_dir: 
        continue
    
    try:
        with open(os.path.join(exp_dir, "results.json"), 'r') as f:
            data = json.load(f)
        df_parquet = pd.read_parquet(os.path.join(exp_dir, "results.parquet"))
        
        sums = df_parquet.groupby('split')[['true_profit', 'real_profit']].sum()
        regret = (sums['true_profit'] - sums['real_profit']) / sums['true_profit']
        regret = (regret * 100).round(1)
        
        all_tsfm_results.append({
            'model': model,
            'train_mse': data.get('final_train_results', {}).get('mse', np.nan),
            'val_mse': data.get('final_val_results', {}).get('mse', np.nan),
            'test_mse': data.get('test_results', {}).get('mse', np.nan),
            'train_profit': sums['real_profit'].get('train', np.nan),
            'val_profit': sums['real_profit'].get('val', np.nan),
            'test_profit': sums['real_profit'].get('test', np.nan),
            'train_regret': regret.get('train', np.nan),
            'val_regret': regret.get('val', np.nan),
            'test_regret': regret.get('test', np.nan)
        })
        # print(f"Successfully processed model: {model}")
    except Exception as e:
        print(f"Error processing model {model}: {e}")
        pass

df_tsfm = pd.DataFrame(all_tsfm_results)
if not df_tsfm.empty:
    df_tsfm = df_tsfm.sort_values('model').reset_index(drop=True)
df_tsfm

,model,train_mse,val_mse,test_mse,train_profit,val_profit,test_profit,train_regret,val_regret,test_regret
0,moment-base-finetuned,923.970520,1229.195068,1449.115356,146406.603864,26413.671214,44977.560394,28.9,54.8,49.7
1,moment-base-finetuned-decision,2105.221680,1628.137573,2356.553955,126308.453042,29078.999367,45721.284891,38.7,50.2,48.8
2,moment-base-lora,910.887207,1229.216675,1451.318970,146658.412301,26632.596690,44939.684315,28.8,54.4,49.7
3,moment-base-lora-decision,2566.955322,1872.331787,2809.929688,118402.755672,28838.034590,45786.650948,42.5,50.6,48.8
4,moment-base-zeroshot,1962.787231,1523.163330,2153.526855,-30278.914802,-12587.958540,-16284.425235,114.7,121.5,118.2
5,moment-large-finetuned,933.712585,1242.135376,1409.101318,146699.733521,26162.859535,45315.934980,28.8,55.2,49.3
6,moment-large-finetuned-decision,2015.442627,1617.128174,2250.712891,122074.583484,28414.765860,44666.976145,40.8,51.4,50.0
7,moment-large-lora,953.742920,1240.238037,1406.375366,145582.246794,26908.548943,46163.586526,29.3,53.9,48.3
8,moment-large-lora-decision,2008.418701,1615.110596,2253.629883,125143.232764,28395.504739,45039.158642,39.3,51.4,49.6
9,moment-large-zeroshot,1918.667480,1483.763428,2087.747070,6807.870293,1308.926374,4189.658114,96.7,97.8,95.3
